## Monopole Antenna

We are going to modify the code of the dipole antenna to create a monopole antenna. 
The monopole antenna is a half of the dipole antenna, and it is typically mounted above a ground plane.

In [ ]:
from palacetoolkit.viz import view_mesh
from palacetoolkit.geometry import extract_tag, xmin, xmax, ymin, ymax, zmin, zmax

import math
import gmsh
import os

Parameters:

- filename   - Output mesh filename
- wavelength - Wavelength of the resulting electromagnetic wave
- arm_length - Length of each antenna arm (default: wavelength/4)
- arm_radius - Radius of the cylindrical antenna arms (default: arm_length/20)
- base_length - Length of the ground plane (default: arm_radius * 4)
- base_width - Width of the ground plane (default: arm_radius * 4)
- gap_size   - Size of the gap between the two arms (port region) (default: arm_length/100)
- outer_boundary_radius - Radius of the outer spherical boundary (default: 1.5*wavelength)
- verbose    - Gmsh verbosity level (0-5, higher = more verbose)
- gui - Whether to launch the Gmsh GUI after mesh generation

In [ ]:
filename = "monopole.msh"
wavelength = 4.0
base_width = None
base_length = None
arm_length = None
arm_radius = None
gap_size = None
outer_boundary_radius = None
verbose = 5
gui = True

# Set default values
if arm_length is None:
    arm_length = wavelength / 4
if arm_radius is None:
    arm_radius = arm_length / 20
if gap_size is None:
    gap_size = arm_length / 100
if outer_boundary_radius is None:
    outer_boundary_radius = 1.5 * wavelength
if base_width is None:
    base_width = arm_radius * 4
if base_length is None:
    base_length = arm_radius * 4

Initialize gmsh

In [ ]:
gmsh.initialize()
kernel = gmsh.model.occ
gmsh.option.setNumber("General.Verbosity", verbose)

# Create a new model. The name dipole is not important. If a model was already added,
# remove it first (this is useful when interactively evaluating this function).
if "monopole" in gmsh.model.list():
    gmsh.model.setCurrent("monopole")
    gmsh.model.remove()
gmsh.model.add("monopole")

In [ ]:
# Mesh refinement parameter: controls elements around cylinder circumference.
# Higher number = higher resolution.
n_circle = 12
# How many elements per wavelength on the outer sphere.
# Higher number = higher resolution.
n_farfield = 3

# Create geometry: outer sphere, the cylindrical arm and the rectangular base.

# addSphere: The first three parameters are the coordinates of the center, and the last one is the radius.
outer_boundary = kernel.addSphere(0, 0, 0, outer_boundary_radius)

# addCylinder: The first three parameters are the coordinates of the center of the cylinder base,
# the second three defines its axis and the last one its radius.
arm = kernel.addCylinder(0, 0, gap_size / 2, 0, 0, arm_length, arm_radius)

# addRectangle: The first three parameters are the coordinates of the left corner of the rectangle, 
# and the next two define the width and length of the rectangle.
base = kernel.addRectangle(-base_width/2, -base_length/2, -gap_size/2,
                            base_width, base_length)


# Create gap rectangle (port region) and rotate to XZ plane (we need to do this because
# OpenCASCADE does not have a primitive to create a rectangle directly on the XZ
# plane...).
gap_rectangle = kernel.addRectangle(-arm_radius, -gap_size/2, 0, 2*arm_radius, gap_size)
kernel.rotate([(2, gap_rectangle)], 0, 0, 0, 1, 0, 0, math.pi/2)

# Fragment geometry to create mesh domain. 
# We fragment the outer sphere with the arm and the base to create the main domain, and we
# fragment the arm and the base with the gap rectangle to create the port region and ensure
# it is properly embedded in the geometry. 
kernel.fragment([(3, outer_boundary)], [(3, arm), (2, base), (2, gap_rectangle)])

# Synchronize CAD operations with Gmsh model.
kernel.synchronize()

Now that we have fragmented the geometry, the tags for the various entities might have
changed, so we find them again using their expected positions/sizes.

In [ ]:
# Helper functions to identify the various components.
all_2d_entities = kernel.getEntities(2)
all_3d_entities = kernel.getEntities(3)

# zmin(x) > 0 means that x is fully above xy plane (similar with zmax(x) < 0).

def find_2D_top_arm():
    return [x for x in all_2d_entities if zmin(x) > 0]

# The only objects that span the entire domain have xmin = - outer_boundary_radius (this
# is true for all the directions).

def spans_domain(x):
    return math.isclose(xmin(x), -outer_boundary_radius, abs_tol=outer_boundary_radius/100)

def find_2D_outer_sphere():
    return [x for x in all_2d_entities if spans_domain(x)]

def find_3D_domain():
    return [x for x in all_3d_entities if spans_domain(x)]

# Helper functions to find 3D arm domains (interior of cylinders)
def find_3D_top_arm():
    return [x for x in all_3d_entities if zmin(x) > 0 and not spans_domain(x)]

def find_base():
    return [x for x in all_2d_entities if zmax(x) < 0]

# We find the rectangle because we know its precise location and size (filling the
# entire gap and being oriented on the XZ plane at y=0).
def is_gap_rectangle(x):
    eps = gap_size / 100
    return (math.isclose(xmin(x), -arm_radius, abs_tol=eps) and
            math.isclose(xmax(x), arm_radius, abs_tol=eps) and
            math.isclose(ymin(x), 0, abs_tol=eps) and
            math.isclose(ymax(x), 0, abs_tol=eps) and
            math.isclose(zmin(x), -gap_size/2, abs_tol=eps) and
            math.isclose(zmax(x), +gap_size/2, abs_tol=eps))

def find_gap_rectangle():
    return [x for x in all_2d_entities if is_gap_rectangle(x)]

# Now we can identify the regions.
outer_sphere_dimtags = find_2D_outer_sphere()
arm_dimtags = find_2D_top_arm()
base_dimtags = find_base()
domain_dimtags = find_3D_domain()
gap_rectangle_dimtags = find_gap_rectangle()
arm_domain_dimtags = find_3D_top_arm()

# Verify we found the expected number of entities.
assert len(arm_dimtags) == 4, f"Expected 4 top arm surfaces, found {len(arm_dimtags)}"
assert len(base_dimtags) == 1, f"Expected 1 base surface, found {len(base_dimtags)}"
assert len(gap_rectangle_dimtags) == 1, f"Expected 1 gap rectangle, found {len(gap_rectangle_dimtags)}"
assert len(outer_sphere_dimtags) == 1, f"Expected 1 outer sphere, found {len(outer_sphere_dimtags)}"
assert len(arm_domain_dimtags) == 1, f"Expected 1 top arm domain, found {len(arm_domain_dimtags)}"
assert len(domain_dimtags) == 1, f"Expected 1 domain, found {len(domain_dimtags)}"

Create physical groups

In [ ]:
# Create physical groups (these become attributes in Palace).
arm_group = gmsh.model.addPhysicalGroup(
    2, [extract_tag(x) for x in arm_dimtags], -1, "top_arm"
)
base_group = gmsh.model.addPhysicalGroup(
    2, [extract_tag(x) for x in base_dimtags], -1, "base"
)
gap_rectangle_group = gmsh.model.addPhysicalGroup(
    2, [extract_tag(x) for x in gap_rectangle_dimtags], -1, "port"
)
outer_boundary_group = gmsh.model.addPhysicalGroup(
    2, [extract_tag(x) for x in outer_sphere_dimtags], -1, "outer_boundary"
)
arm_domain_group = gmsh.model.addPhysicalGroup(
    3, [extract_tag(x) for x in arm_domain_dimtags], -1, "top_arm_domain"
)
domain_group = gmsh.model.addPhysicalGroup(
    3, [extract_tag(x) for x in domain_dimtags], -1, "domain"
)

Mesh size parameters and mesh generation

In [ ]:
# Set mesh size parameters.
gmsh.option.setNumber("Mesh.MeshSizeMin", 2.0 * math.pi * arm_radius / n_circle / 2.0)
gmsh.option.setNumber("Mesh.MeshSizeMax", wavelength / n_farfield)
# Set minimum number of elements per 2π radians of curvature.
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", n_circle)
# Don't extend mesh size constraints from boundaries into the volume.
# This option is typically activated when working with mesh size fields.
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)

# Finally, we control mesh size using a mesh size field.

# First, create a mesh size field (with id 1) that extends from the inner surfaces to
# the volume toward the outer boundary.
gmsh.model.mesh.field.add("Extend", 1)
gmsh.model.mesh.field.setNumbers(
    1,
    "SurfacesList",
    [extract_tag(x) for x in arm_dimtags + base_dimtags + gap_rectangle_dimtags]
)
gmsh.model.mesh.field.setNumber(1, "DistMax", outer_boundary_radius)
gmsh.model.mesh.field.setNumber(1, "SizeMax", wavelength / n_farfield)

# Finally, use this Extend field to determine element sizes.
gmsh.model.mesh.field.setAsBackgroundMesh(1)

# Set 2D/3D meshing algorithm. Chosen to be deterministic, not necessarily the best.
gmsh.option.setNumber("Mesh.Algorithm3D", 1)
gmsh.option.setNumber("Mesh.Algorithm", 6)

# Generate 3D volume mesh and set to 3rd order elements.
gmsh.model.mesh.generate(3)
gmsh.model.mesh.setOrder(1)

# Set output format for Palace compatibility.
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.option.setNumber("Mesh.Binary", 1)

# Get the directory of this script
script_dir = os.getcwd()
output_path = os.path.join(script_dir, filename)
gmsh.write(output_path)

print("\nFinished generating mesh. Physical group tags:")
print(f"Arm (2D): {arm_group}")
print(f"Base (2D): {base_group}")
print(f"Rectangular port (2D): {gap_rectangle_group}")
print(f"Farfield boundary (2D): {outer_boundary_group}")
print(f"Arm domain (3D): {arm_domain_group}")
print(f"Domain (3D): {domain_group}")
print()

# Optionally launch the Gmsh GUI.
if gui:
    gmsh.fltk.run()

# Clean up Gmsh resources.
gmsh.finalize()